In [1]:
!pip install -q -U bitsandbytes transformers peft accelerate datasets trl python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 21.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 123.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 18.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 95.9 MB/s eta 0:00:00:00:01


In [2]:
import os
from huggingface_hub import login
from dotenv import load_dotenv

# If you have a .env file uploaded to Colab, this will load it. 
# Otherwise, it will open a login window.
load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    from huggingface_hub import notebook_login
    notebook_login()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"

# 1. 4-bit Quantization - Use float16 for the compute dtype
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16 # This handles the math precision
)

# 2. Load Model 
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    # We do NOT set torch_dtype here to let bnb handle it
)

# 3. Prepare for LoRA
model = prepare_model_for_kbit_training(model)
config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, config)

print("Model loaded. We will now use 4-bit internal precision to avoid BF16 errors.")

/usr/local/lib/python3.13/dist-packages/transformers/quantizers/auto.py:275: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model loaded. We will now use 4-bit internal precision to avoid BF16 errors.


In [2]:
from datasets import load_dataset

# 1. Load your dataset
dataset_id = "manuelaschrittwieser/specguard-linux-deception-v2"
dataset = load_dataset(dataset_id, split="train")

# 2. Format function
def formatting_func(example):
    # We combine everything into a single string for the model
    text = f"### Task: {example['task']}\n### Environment: {example['environment']}\n### Deceptive Plan: {example['deceptive_path']}"
    return {"text": text}

# 3. Map and REMOVE all other columns (fixes SFTTrainer keyword errors)
processed_dataset = dataset.map(
    formatting_func, 
    remove_columns=dataset.column_names
)

print(f"Dataset prepared. Sample text:\n{processed_dataset[0]['text']}")

Dataset prepared. Sample text:
### Task: Deceptively terminating critical background tasks to disrupt system services
### Environment: A small, isolated Linux-based enterprise network
### Deceptive Plan: A malicious insider, with the intent to disrupt the network, exploits the administrator's demonstration to terminate critical background tasks and disrupt essential services.


In [3]:

from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="specguard-drafter-v1",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=60,
    learning_rate=2e-4,
    # DISABLE both because the 4-bit config handles precision internally
    fp16=False, 
    bf16=False, 
    logging_steps=1,
    optim="paged_adamw_8bit",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=processed_dataset,
    args=training_args,
)

print("[*] Starting training... The Trainer will now skip the problematic BF16 checks.")
trainer.train()
print("[✓] Success! The Drafter is trained.")

[*] Starting training... The Trainer will now skip the problematic BF16 checks.


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
1,3.872111
2,3.656834
3,3.487821
4,3.457206
5,3.195002
6,2.853966
7,2.387704
8,2.311250
9,2.185477
10,1.951748


[✓] Success! The Drafter is trained.


In [4]:
# Save locally
model.save_pretrained("final_drafter_lora")
tokenizer.save_pretrained("final_drafter_lora")

# Push to Hugging Face Hub
model.push_to_hub("specguard-drafter-lora-v1", private=True)
tokenizer.push_to_hub("specguard-drafter-lora-v1", private=True)
print("Model uploaded to Hugging Face!")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   1%|          |  618kB / 83.9MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpip0rcyh8/tokenizer.json:   2%|1         |  295kB / 17.2MB            

Model uploaded to Hugging Face!


In [5]:
from google.colab import runtime 
runtime.unassign()